In [1]:
# # PHASE 05.04 — CROSS-MODAL INFERENCE

# ## Objective

# Apply the frozen Phase 03 baseline cross-modal relationship model
# to validated 32-D modality representations.

# Pipeline:

# 32-D Source Representation
#         ↓
# 32-D Target Representation
#         ↓
# Pairwise Feature Construction
# [S, T, |S-T|, S×T]
#         ↓
# 128-D Pair Representation
#         ↓
# Frozen Phase 03 Baseline
# 128 → 128 → 64 → 1
#         ↓
# Sigmoid
#         ↓
# Cross-Modal Relationship Probability

# No training is performed.
# No checkpoint is modified.
# No patient-level matching is assumed.

# The output represents the probability that a cross-modal pair
# has the same class relationship under the Phase 03 baseline.

In [2]:
import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

print("Libraries imported successfully.")
print("PyTorch version:", torch.__version__)
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)

Libraries imported successfully.
PyTorch version: 2.14.0+cpu
NumPy version: 2.2.6
Pandas version: 2.2.3


In [3]:
PROJECT_ROOT = r"C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening"

PHASE05_ROOT = os.path.join(
    PROJECT_ROOT,
    "06_cross_modal_learning",
    "05_multimodal_integration"
)

NOTEBOOK_ROOT = os.path.join(
    PHASE05_ROOT,
    "notebooks"
)

REPORT_ROOT = os.path.join(
    PHASE05_ROOT,
    "reports",
    "05_04_cross_modal_inference"
)

os.makedirs(REPORT_ROOT, exist_ok=True)

print("PROJECT_ROOT:")
print(PROJECT_ROOT)

print("\nPHASE05_ROOT:")
print(PHASE05_ROOT)

print("\nREPORT_ROOT:")
print(REPORT_ROOT)

PROJECT_ROOT:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening

PHASE05_ROOT:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\05_multimodal_integration

REPORT_ROOT:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\05_multimodal_integration\reports\05_04_cross_modal_inference


In [4]:
GENETIC_32D_PATH = os.path.join(
    PHASE05_ROOT,
    "reports",
    "05_03_common_projection",
    "05_03_genetic_32d_projection.csv"
)

BEHAVIORAL_32D_PATH = os.path.join(
    PHASE05_ROOT,
    "reports",
    "05_03_common_projection",
    "05_03_behavioral_32d_projection.csv"
)

ENVIRONMENTAL_32D_PATH = os.path.join(
    PHASE05_ROOT,
    "reports",
    "05_03_common_projection",
    "05_03_environmental_32d_projection.csv"
)

required_32d_files = {
    "Genetic": GENETIC_32D_PATH,
    "Behavioral": BEHAVIORAL_32D_PATH,
    "Environmental": ENVIRONMENTAL_32D_PATH
}

for name, path in required_32d_files.items():
    exists = os.path.exists(path)
    print(f"{name:15s}: {'FOUND' if exists else 'MISSING'}")
    print(path)
    
    if not exists:
        raise FileNotFoundError(path)

Genetic        : FOUND
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\05_multimodal_integration\reports\05_03_common_projection\05_03_genetic_32d_projection.csv
Behavioral     : FOUND
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\05_multimodal_integration\reports\05_03_common_projection\05_03_behavioral_32d_projection.csv
Environmental  : FOUND
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\05_multimodal_integration\reports\05_03_common_projection\05_03_environmental_32d_projection.csv


In [5]:
genetic_32d = pd.read_csv(GENETIC_32D_PATH)
behavioral_32d = pd.read_csv(BEHAVIORAL_32D_PATH)
environmental_32d = pd.read_csv(ENVIRONMENTAL_32D_PATH)

print("Genetic shape:", genetic_32d.shape)
print("Behavioral shape:", behavioral_32d.shape)
print("Environmental shape:", environmental_32d.shape)

Genetic shape: (5, 33)
Behavioral shape: (5, 33)
Environmental shape: (5, 33)


In [6]:
def verify_32d_dataframe(df, name):
    if len(df.columns) != 33:
        raise ValueError(
            f"{name}: expected 33 columns "
            f"(1 ID + 32 features), found {len(df.columns)}"
        )

    if df.shape[1] - 1 != 32:
        raise ValueError(
            f"{name}: expected 32 representation dimensions."
        )

    feature_cols = list(df.columns[1:])

    values = df[feature_cols].apply(
        pd.to_numeric,
        errors="raise"
    ).values.astype(np.float32)

    if np.isnan(values).any():
        raise ValueError(f"{name}: NaN detected.")

    if np.isinf(values).any():
        raise ValueError(f"{name}: infinite value detected.")

    if df.iloc[:, 0].duplicated().any():
        raise ValueError(f"{name}: duplicate sample IDs detected.")

    return feature_cols


genetic_cols = verify_32d_dataframe(
    genetic_32d,
    "Genetic"
)

behavioral_cols = verify_32d_dataframe(
    behavioral_32d,
    "Behavioral"
)

environmental_cols = verify_32d_dataframe(
    environmental_32d,
    "Environmental"
)

print("32-D representation validation: PASSED")

32-D representation validation: PASSED


In [7]:
genetic_ids = list(genetic_32d.iloc[:, 0])
behavioral_ids = list(behavioral_32d.iloc[:, 0])
environmental_ids = list(environmental_32d.iloc[:, 0])

if genetic_ids != behavioral_ids:
    raise ValueError(
        "Genetic and Behavioral sample IDs are not aligned."
    )

if genetic_ids != environmental_ids:
    raise ValueError(
        "Genetic and Environmental sample IDs are not aligned."
    )

print("Shared demonstration IDs:")
print(genetic_ids)

print("\nPatient/sample ID alignment: PASSED")

Shared demonstration IDs:
['P001', 'P002', 'P003', 'P004', 'P005']

Patient/sample ID alignment: PASSED


In [8]:
BASELINE_CHECKPOINT_PATH = os.path.join(
    PROJECT_ROOT,
    "06_cross_modal_learning",
    "03_cross_modal_learning",
    "03_04_baseline_training",
    "checkpoints",
    "03_04_baseline_best_model.pt"
)

if not os.path.exists(BASELINE_CHECKPOINT_PATH):
    raise FileNotFoundError(
        f"Baseline checkpoint not found:\n"
        f"{BASELINE_CHECKPOINT_PATH}"
    )

print("Baseline checkpoint found:")
print(BASELINE_CHECKPOINT_PATH)

Baseline checkpoint found:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\03_cross_modal_learning\03_04_baseline_training\checkpoints\03_04_baseline_best_model.pt


In [9]:
checkpoint = torch.load(
    BASELINE_CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=True
)

print("Checkpoint type:", type(checkpoint))

if not isinstance(checkpoint, dict):
    raise TypeError("Unexpected checkpoint format.")

print("\nCheckpoint keys:")
for key in checkpoint.keys():
    print(" -", key)

Checkpoint type: <class 'dict'>

Checkpoint keys:
 - epoch
 - model_state_dict
 - optimizer_state_dict
 - best_val_roc_auc
 - seed
 - batch_size
 - learning_rate
 - num_epochs


In [10]:
state_dict = checkpoint["model_state_dict"]

expected_shapes = {
    "classifier.0.weight": (128, 128),
    "classifier.0.bias": (128,),
    "classifier.2.weight": (64, 128),
    "classifier.2.bias": (64,),
    "classifier.4.weight": (1, 64),
    "classifier.4.bias": (1,)
}

for key, expected_shape in expected_shapes.items():
    if key not in state_dict:
        raise KeyError(
            f"Missing checkpoint parameter: {key}"
        )

    actual_shape = tuple(state_dict[key].shape)

    if actual_shape != expected_shape:
        raise ValueError(
            f"{key}: expected {expected_shape}, "
            f"found {actual_shape}"
        )

print("Exact Phase 03 baseline checkpoint structure: PASSED")

Exact Phase 03 baseline checkpoint structure: PASSED


In [11]:
class CrossModalBaseline(nn.Module):

    def __init__(self):
        super().__init__()

        self.classifier = nn.Sequential(
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.classifier(x)


baseline_model = CrossModalBaseline()

baseline_model.load_state_dict(state_dict)

baseline_model.eval()

for parameter in baseline_model.parameters():
    parameter.requires_grad = False

print(baseline_model)

CrossModalBaseline(
  (classifier): Sequential(
    (0): Linear(in_features=128, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=1, bias=True)
  )
)


In [12]:
total_parameters = sum(
    parameter.numel()
    for parameter in baseline_model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in baseline_model.parameters()
    if parameter.requires_grad
)

print("Total parameters:", total_parameters)
print("Trainable parameters:", trainable_parameters)

if total_parameters != 24833:
    raise ValueError(
        f"Unexpected parameter count: {total_parameters}"
    )

if trainable_parameters != 0:
    raise ValueError(
        "Baseline model is not completely frozen."
    )

print("\nBaseline model verification: PASSED")
print("No training will be performed.")

Total parameters: 24833
Trainable parameters: 0

Baseline model verification: PASSED
No training will be performed.


In [13]:
def construct_pair_features(source, target):

    source = np.asarray(
        source,
        dtype=np.float32
    )

    target = np.asarray(
        target,
        dtype=np.float32
    )

    if source.shape != target.shape:
        raise ValueError(
            f"Source and target shapes differ: "
            f"{source.shape} vs {target.shape}"
        )

    if source.ndim != 2:
        raise ValueError(
            "Source and target must be 2-D arrays."
        )

    if source.shape[1] != 32:
        raise ValueError(
            f"Expected 32-D representations, "
            f"found {source.shape[1]}"
        )

    absolute_difference = np.abs(
        source - target
    )

    elementwise_product = (
        source * target
    )

    pair_features = np.concatenate(
        [
            source,
            target,
            absolute_difference,
            elementwise_product
        ],
        axis=1
    )

    return pair_features


test_source = np.zeros((2, 32), dtype=np.float32)
test_target = np.ones((2, 32), dtype=np.float32)

test_pair = construct_pair_features(
    test_source,
    test_target
)

print("Test pair shape:", test_pair.shape)

if test_pair.shape != (2, 128):
    raise ValueError(
        "Pair feature construction failed."
    )

print("Pair feature construction: PASSED")

Test pair shape: (2, 128)
Pair feature construction: PASSED


In [16]:
#temp check

SEARCH_ROOT = os.path.join(
    PROJECT_ROOT,
    "06_cross_modal_learning"
)

print("Searching for possible Phase 02 pair artifacts...\n")

for root, dirs, files in os.walk(SEARCH_ROOT):

    for filename in files:

        lower_name = filename.lower()

        if (
            "pair" in lower_name
            or "cross_modal" in lower_name
            or "02_08" in lower_name
        ):

            full_path = os.path.join(
                root,
                filename
            )

            print(full_path)

Searching for possible Phase 02 pair artifacts...

C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\01_embedding_analysis\figures\01_07_overall_cross_modal_similarity_distribution.png
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\01_embedding_analysis\notebooks\01_07_cross_modal_similarity.ipynb
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\01_embedding_analysis\notebooks\.ipynb_checkpoints\01_07_cross_modal_similarity-checkpoint.ipynb
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\01_embedding_analysis\reports\01_07_cross_modal_similarity_final_summary.csv
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\01_embedding_analysis\reports\01_07_overall_cross_modal_similarity.csv
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_moda

In [17]:
PHASE02_PAIR_ROOT = os.path.join(
    PROJECT_ROOT,
    "06_cross_modal_learning",
    "02_cross_modal_pairing",
    "pairs",
    "representation_aware_split"
)

VALIDATION_PAIR_NPZ = os.path.join(
    PHASE02_PAIR_ROOT,
    "02_08_validation_representation_disjoint_pairs.npz"
)

VALIDATION_PAIR_METADATA = os.path.join(
    PHASE02_PAIR_ROOT,
    "02_08_validation_representation_disjoint_metadata.csv"
)

required_pair_files = {
    "Validation pair data": VALIDATION_PAIR_NPZ,
    "Validation pair metadata": VALIDATION_PAIR_METADATA
}

print("=" * 70)
print("PHASE 02.08 VALIDATION PAIR ARTIFACTS")
print("=" * 70)

for name, path in required_pair_files.items():

    if not os.path.exists(path):
        raise FileNotFoundError(
            f"{name} not found:\n{path}"
        )

    print(f"\n[PASSED] {name}")
    print(path)

print("\nPhase 02.08 validation artifacts: FOUND")

PHASE 02.08 VALIDATION PAIR ARTIFACTS

[PASSED] Validation pair data
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\pairs\representation_aware_split\02_08_validation_representation_disjoint_pairs.npz

[PASSED] Validation pair metadata
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\pairs\representation_aware_split\02_08_validation_representation_disjoint_metadata.csv

Phase 02.08 validation artifacts: FOUND


In [18]:
validation_metadata = pd.read_csv(
    VALIDATION_PAIR_METADATA
)

print("=" * 70)
print("VALIDATION PAIR METADATA")
print("=" * 70)

print("\nShape:")
print(validation_metadata.shape)

print("\nColumns:")
for column in validation_metadata.columns:
    print(" -", column)

print("\nFirst 5 rows:")
display(validation_metadata.head())

VALIDATION PAIR METADATA

Shape:
(448, 8)

Columns:
 - source_modality
 - target_modality
 - source_index
 - target_index
 - source_class
 - target_class
 - pair_label
 - pair_type

First 5 rows:


,source_modality,target_modality,source_index,target_index,source_class,target_class,pair_label,pair_type
0,Genetic,Behavioral,10,693,0,0,1,positive
1,Genetic,Behavioral,14,702,0,0,1,positive
2,Genetic,Behavioral,20,635,0,0,1,positive
3,Genetic,Behavioral,6,811,0,0,1,positive
4,Genetic,Behavioral,19,158,0,0,1,positive


In [19]:
validation_pairs = np.load(
    VALIDATION_PAIR_NPZ,
    allow_pickle=True
)

print("=" * 70)
print("VALIDATION PAIR NPZ CONTENT")
print("=" * 70)

print("\nAvailable arrays:")

for key in validation_pairs.files:
    array = validation_pairs[key]

    print(
        f"{key:30s} "
        f"shape={array.shape} "
        f"dtype={array.dtype}"
    )

VALIDATION PAIR NPZ CONTENT

Available arrays:
source_embeddings              shape=(448, 32) dtype=float32
target_embeddings              shape=(448, 32) dtype=float32
pair_labels                    shape=(448,) dtype=int64


In [20]:
print("=" * 70)
print("VALIDATION DATASET SIZE")
print("=" * 70)

for key in validation_pairs.files:

    array = validation_pairs[key]

    print(
        f"{key}: {array.shape}"
    )

VALIDATION DATASET SIZE
source_embeddings: (448, 32)
target_embeddings: (448, 32)
pair_labels: (448,)


In [21]:
metadata_count = len(validation_metadata)

print("Validation metadata rows:", metadata_count)

if metadata_count != 448:
    raise ValueError(
        f"Expected 448 validation metadata rows, "
        f"found {metadata_count}"
    )

print("Validation metadata count: PASSED")

Validation metadata rows: 448
Validation metadata count: PASSED


In [22]:
# ============================================================
# CELL 19 — EXTRACT VALIDATION PAIR ARRAYS
# ============================================================

source_embeddings = validation_pairs["source_embeddings"]
target_embeddings = validation_pairs["target_embeddings"]
pair_labels = validation_pairs["pair_labels"]

print("=" * 70)
print("VALIDATION PAIR ARRAYS")
print("=" * 70)

print("\nSource embeddings:")
print("Shape:", source_embeddings.shape)
print("Dtype:", source_embeddings.dtype)

print("\nTarget embeddings:")
print("Shape:", target_embeddings.shape)
print("Dtype:", target_embeddings.dtype)

print("\nPair labels:")
print("Shape:", pair_labels.shape)
print("Dtype:", pair_labels.dtype)

print("\nValidation pair arrays: LOADED")

VALIDATION PAIR ARRAYS

Source embeddings:
Shape: (448, 32)
Dtype: float32

Target embeddings:
Shape: (448, 32)
Dtype: float32

Pair labels:
Shape: (448,)
Dtype: int64

Validation pair arrays: LOADED


In [23]:
# ============================================================
# CELL 20 — VALIDATE VALIDATION PAIR DATA
# ============================================================

if source_embeddings.shape != (448, 32):
    raise ValueError(
        f"Unexpected source embedding shape: {source_embeddings.shape}"
    )

if target_embeddings.shape != (448, 32):
    raise ValueError(
        f"Unexpected target embedding shape: {target_embeddings.shape}"
    )

if pair_labels.shape != (448,):
    raise ValueError(
        f"Unexpected pair label shape: {pair_labels.shape}"
    )

if not np.isfinite(source_embeddings).all():
    raise ValueError("Source embeddings contain NaN or infinite values.")

if not np.isfinite(target_embeddings).all():
    raise ValueError("Target embeddings contain NaN or infinite values.")

unique_labels = np.unique(pair_labels)

print("=" * 70)
print("VALIDATION PAIR INTEGRITY")
print("=" * 70)

print("\nUnique labels:", unique_labels)

for label in unique_labels:
    count = np.sum(pair_labels == label)
    print(f"Label {label}: {count} pairs")

if not np.array_equal(unique_labels, np.array([0, 1])):
    raise ValueError(
        f"Expected labels [0, 1], found {unique_labels}"
    )

print("\nValidation pair integrity: PASSED")

VALIDATION PAIR INTEGRITY

Unique labels: [0 1]
Label 0: 224 pairs
Label 1: 224 pairs

Validation pair integrity: PASSED


In [24]:
# ============================================================
# CELL 21 — CONSTRUCT 128-D BASELINE INPUT
# ============================================================

pair_features = construct_pair_features(
    source_embeddings,
    target_embeddings
)

print("=" * 70)
print("BASELINE PAIR FEATURE CONSTRUCTION")
print("=" * 70)

print("\nSource dimension:", source_embeddings.shape[1])
print("Target dimension:", target_embeddings.shape[1])

print("\nConstructed pair feature shape:")
print(pair_features.shape)

expected_shape = (448, 128)

if pair_features.shape != expected_shape:
    raise ValueError(
        f"Expected pair feature shape {expected_shape}, "
        f"found {pair_features.shape}"
    )

if not np.isfinite(pair_features).all():
    raise ValueError(
        "Constructed pair features contain NaN or infinite values."
    )

print("\nExpected components:")
print("Source:              32")
print("Target:              32")
print("Absolute difference: 32")
print("Elementwise product: 32")
print("---------------------------")
print("Total:              128")

print("\n128-D pair feature construction: PASSED")

BASELINE PAIR FEATURE CONSTRUCTION

Source dimension: 32
Target dimension: 32

Constructed pair feature shape:
(448, 128)

Expected components:
Source:              32
Target:              32
Absolute difference: 32
Elementwise product: 32
---------------------------
Total:              128

128-D pair feature construction: PASSED


In [25]:
# ============================================================
# CELL 22 — CONVERT PAIR FEATURES TO PYTORCH TENSOR
# ============================================================

pair_features_tensor = torch.tensor(
    pair_features,
    dtype=torch.float32
)

print("=" * 70)
print("PYTORCH BASELINE INPUT")
print("=" * 70)

print("\nTensor shape:", pair_features_tensor.shape)
print("Tensor dtype:", pair_features_tensor.dtype)
print("Device:", pair_features_tensor.device)

if pair_features_tensor.shape != (448, 128):
    raise ValueError(
        f"Unexpected tensor shape: {pair_features_tensor.shape}"
    )

print("\nPyTorch input preparation: PASSED")

PYTORCH BASELINE INPUT

Tensor shape: torch.Size([448, 128])
Tensor dtype: torch.float32
Device: cpu

PyTorch input preparation: PASSED


In [26]:
# ============================================================
# CELL 23 — FROZEN CROSS-MODAL BASELINE INFERENCE
# ============================================================

with torch.no_grad():
    logits = baseline_model(pair_features_tensor)

print("=" * 70)
print("CROSS-MODAL BASELINE INFERENCE")
print("=" * 70)

print("\nLogit tensor shape:", logits.shape)
print("Logit tensor dtype:", logits.dtype)

if logits.shape != (448, 1):
    raise ValueError(
        f"Unexpected logit shape: {logits.shape}"
    )

print("\nFrozen baseline inference: PASSED")

CROSS-MODAL BASELINE INFERENCE

Logit tensor shape: torch.Size([448, 1])
Logit tensor dtype: torch.float32

Frozen baseline inference: PASSED


In [27]:
# ============================================================
# CELL 24 — CONVERT LOGITS TO PROBABILITIES
# ============================================================

probabilities = torch.sigmoid(logits).squeeze(1)

probabilities_np = probabilities.cpu().numpy()

print("=" * 70)
print("CROSS-MODAL RELATIONSHIP PROBABILITIES")
print("=" * 70)

print("\nProbability shape:", probabilities_np.shape)

print("Minimum probability:", float(probabilities_np.min()))
print("Maximum probability:", float(probabilities_np.max()))
print("Mean probability:", float(probabilities_np.mean()))

if probabilities_np.shape != (448,):
    raise ValueError(
        f"Unexpected probability shape: {probabilities_np.shape}"
    )

if not np.isfinite(probabilities_np).all():
    raise ValueError(
        "Probabilities contain NaN or infinite values."
    )

if probabilities_np.min() < 0 or probabilities_np.max() > 1:
    raise ValueError(
        "Probabilities are outside the [0, 1] range."
    )

print("\nProbability conversion: PASSED")

CROSS-MODAL RELATIONSHIP PROBABILITIES

Probability shape: (448,)
Minimum probability: 0.03269292041659355
Maximum probability: 0.9379035830497742
Mean probability: 0.5153837203979492

Probability conversion: PASSED


In [28]:
# ============================================================
# CELL 25 — BUILD INFERENCE RESULT TABLE
# ============================================================

predicted_labels = (
    probabilities_np >= 0.50
).astype(int)

inference_results = pd.DataFrame({
    "pair_index": np.arange(len(pair_labels)),
    "true_label": pair_labels,
    "logit": logits.squeeze(1).cpu().numpy(),
    "relationship_probability": probabilities_np,
    "predicted_label": predicted_labels
})

print("=" * 70)
print("INFERENCE RESULT TABLE")
print("=" * 70)

print("\nShape:", inference_results.shape)

print("\nFirst 5 predictions:")
display(inference_results.head())

print("\nPrediction label counts:")
print(
    inference_results["predicted_label"]
    .value_counts()
    .sort_index()
)

print("\nInference result table: CREATED")

INFERENCE RESULT TABLE

Shape: (448, 5)

First 5 predictions:


,pair_index,true_label,logit,relationship_probability,predicted_label
0,0,1,0.201743,0.550265,1
1,1,1,-1.208289,0.230004,0
2,2,1,0.048158,0.512037,1
3,3,1,-1.454747,0.189272,0
4,4,1,-0.749759,0.320874,0



Prediction label counts:
predicted_label
0    214
1    234
Name: count, dtype: int64

Inference result table: CREATED


In [29]:
# ============================================================
# CELL 26 — CALCULATE CROSS-MODAL INFERENCE METRICS
# ============================================================

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

y_true = pair_labels
y_probability = probabilities_np
y_pred = predicted_labels

roc_auc = roc_auc_score(
    y_true,
    y_probability
)

accuracy = accuracy_score(
    y_true,
    y_pred
)

balanced_accuracy = balanced_accuracy_score(
    y_true,
    y_pred
)

precision = precision_score(
    y_true,
    y_pred,
    zero_division=0
)

recall = recall_score(
    y_true,
    y_pred,
    zero_division=0
)

f1 = f1_score(
    y_true,
    y_pred,
    zero_division=0
)

cm = confusion_matrix(
    y_true,
    y_pred
)

print("=" * 70)
print("CROSS-MODAL INFERENCE METRICS")
print("=" * 70)

print(f"\nROC-AUC:             {roc_auc:.10f}")
print(f"Accuracy:            {accuracy:.10f}")
print(f"Balanced Accuracy:   {balanced_accuracy:.10f}")
print(f"Precision:            {precision:.10f}")
print(f"Recall:               {recall:.10f}")
print(f"F1 Score:             {f1:.10f}")

print("\nConfusion Matrix:")
print(cm)

CROSS-MODAL INFERENCE METRICS

ROC-AUC:             0.6626674107
Accuracy:            0.6116071429
Balanced Accuracy:   0.6116071429
Precision:            0.6068376068
Recall:               0.6339285714
F1 Score:             0.6200873362

Confusion Matrix:
[[132  92]
 [ 82 142]]


In [30]:
# ============================================================
# CELL 27 — INSPECT VALIDATION PAIR METADATA
# ============================================================

print("=" * 70)
print("VALIDATION PAIR METADATA INSPECTION")
print("=" * 70)

print("\nColumns:")
for column in validation_metadata.columns:
    print(" -", column)

print("\nData types:")
print(validation_metadata.dtypes)

print("\nFirst 10 rows:")
display(validation_metadata.head(10))

VALIDATION PAIR METADATA INSPECTION

Columns:
 - source_modality
 - target_modality
 - source_index
 - target_index
 - source_class
 - target_class
 - pair_label
 - pair_type

Data types:
source_modality    object
target_modality    object
source_index        int64
target_index        int64
source_class        int64
target_class        int64
pair_label          int64
pair_type          object
dtype: object

First 10 rows:


,source_modality,target_modality,source_index,target_index,source_class,target_class,pair_label,pair_type
0,Genetic,Behavioral,10,693,0,0,1,positive
1,Genetic,Behavioral,14,702,0,0,1,positive
2,Genetic,Behavioral,20,635,0,0,1,positive
3,Genetic,Behavioral,6,811,0,0,1,positive
4,Genetic,Behavioral,19,158,0,0,1,positive
5,Genetic,Behavioral,7,845,0,0,1,positive
6,Genetic,Behavioral,92,228,1,1,1,positive
7,Genetic,Behavioral,108,133,1,1,1,positive
8,Genetic,Behavioral,70,753,1,1,1,positive
9,Genetic,Behavioral,67,394,1,1,1,positive


In [31]:
# ============================================================
# CELL 28 — VERIFY METADATA / PREDICTION ALIGNMENT
# ============================================================

print("=" * 70)
print("METADATA / PREDICTION ALIGNMENT")
print("=" * 70)

if len(validation_metadata) != len(inference_results):
    raise ValueError(
        "Metadata row count does not match prediction count."
    )

if not np.array_equal(
    validation_metadata["pair_label"].to_numpy(),
    pair_labels
):
    raise ValueError(
        "Metadata pair_label does not match NPZ pair_labels."
    )

if not np.array_equal(
    validation_metadata["pair_label"].to_numpy(),
    inference_results["true_label"].to_numpy()
):
    raise ValueError(
        "Metadata labels do not match inference labels."
    )

print("\nMetadata rows:", len(validation_metadata))
print("Prediction rows:", len(inference_results))

print("\nPair labels match NPZ labels: PASSED")
print("Metadata / prediction alignment: PASSED")

METADATA / PREDICTION ALIGNMENT

Metadata rows: 448
Prediction rows: 448

Pair labels match NPZ labels: PASSED
Metadata / prediction alignment: PASSED


In [32]:
# ============================================================
# CELL 29 — COMBINE METADATA WITH MODEL PREDICTIONS
# ============================================================

prediction_table = validation_metadata.copy()

prediction_table["pair_index"] = np.arange(
    len(prediction_table)
)

prediction_table["logit"] = (
    logits.squeeze(1)
    .cpu()
    .numpy()
)

prediction_table["relationship_probability"] = (
    probabilities_np
)

prediction_table["predicted_label"] = (
    predicted_labels
)

print("=" * 70)
print("FINAL CROSS-MODAL PREDICTION TABLE")
print("=" * 70)

print("\nShape:", prediction_table.shape)

print("\nColumns:")
for column in prediction_table.columns:
    print(" -", column)

print("\nFirst 5 rows:")
display(prediction_table.head())

FINAL CROSS-MODAL PREDICTION TABLE

Shape: (448, 12)

Columns:
 - source_modality
 - target_modality
 - source_index
 - target_index
 - source_class
 - target_class
 - pair_label
 - pair_type
 - pair_index
 - logit
 - relationship_probability
 - predicted_label

First 5 rows:


,source_modality,target_modality,source_index,target_index,source_class,target_class,pair_label,pair_type,pair_index,logit,relationship_probability,predicted_label
0,Genetic,Behavioral,10,693,0,0,1,positive,0,0.201743,0.550265,1
1,Genetic,Behavioral,14,702,0,0,1,positive,1,-1.208289,0.230004,0
2,Genetic,Behavioral,20,635,0,0,1,positive,2,0.048158,0.512037,1
3,Genetic,Behavioral,6,811,0,0,1,positive,3,-1.454747,0.189272,0
4,Genetic,Behavioral,19,158,0,0,1,positive,4,-0.749759,0.320874,0


In [33]:
# ============================================================
# CELL 30 — IDENTIFY MODALITY COMBINATIONS
# ============================================================

prediction_table["modality_combination"] = (
    prediction_table["source_modality"]
    + "-"
    + prediction_table["target_modality"]
)

print("=" * 70)
print("MODALITY COMBINATIONS")
print("=" * 70)

combination_counts = (
    prediction_table["modality_combination"]
    .value_counts()
    .sort_index()
)

print("\nPair counts:")
print(combination_counts)

expected_combinations = {
    "Behavioral-Environmental",
    "Genetic-Behavioral",
    "Genetic-Environmental"
}

actual_combinations = set(
    combination_counts.index
)

if actual_combinations != expected_combinations:
    raise ValueError(
        f"Unexpected modality combinations.\n"
        f"Expected: {expected_combinations}\n"
        f"Found: {actual_combinations}"
    )

print("\nAll three expected modality combinations: PASSED")

MODALITY COMBINATIONS

Pair counts:
modality_combination
Behavioral-Environmental    352
Genetic-Behavioral           48
Genetic-Environmental        48
Name: count, dtype: int64

All three expected modality combinations: PASSED


In [34]:
# ============================================================
# CELL 31 — CHECK CLASS DISTRIBUTION BY MODALITY
# ============================================================

combination_class_counts = pd.crosstab(
    prediction_table["modality_combination"],
    prediction_table["pair_label"]
)

print("=" * 70)
print("CLASS DISTRIBUTION BY MODALITY COMBINATION")
print("=" * 70)

display(combination_class_counts)

print("\nExpected columns:")
print([0, 1])

if not set(combination_class_counts.columns).issubset({0, 1}):
    raise ValueError(
        "Unexpected pair labels found."
    )

CLASS DISTRIBUTION BY MODALITY COMBINATION


pair_label,0,1
modality_combination,,
Behavioral-Environmental,176,176
Genetic-Behavioral,24,24
Genetic-Environmental,24,24



Expected columns:
[0, 1]


In [35]:
# ============================================================
# CELL 32 — MODALITY-SPECIFIC CROSS-MODAL METRICS
# ============================================================

modality_results = []

for combination in sorted(
    prediction_table["modality_combination"].unique()
):
    subset = prediction_table[
        prediction_table["modality_combination"] == combination
    ]

    y_true_subset = subset["pair_label"].to_numpy()
    y_prob_subset = subset["relationship_probability"].to_numpy()
    y_pred_subset = subset["predicted_label"].to_numpy()

    cm_subset = confusion_matrix(
        y_true_subset,
        y_pred_subset,
        labels=[0, 1]
    )

    auc_subset = roc_auc_score(
        y_true_subset,
        y_prob_subset
    )

    accuracy_subset = accuracy_score(
        y_true_subset,
        y_pred_subset
    )

    balanced_accuracy_subset = balanced_accuracy_score(
        y_true_subset,
        y_pred_subset
    )

    precision_subset = precision_score(
        y_true_subset,
        y_pred_subset,
        zero_division=0
    )

    recall_subset = recall_score(
        y_true_subset,
        y_pred_subset,
        zero_division=0
    )

    f1_subset = f1_score(
        y_true_subset,
        y_pred_subset,
        zero_division=0
    )

    modality_results.append({
        "modality_combination": combination,
        "n_pairs": len(subset),
        "class_0": int(np.sum(y_true_subset == 0)),
        "class_1": int(np.sum(y_true_subset == 1)),
        "roc_auc": auc_subset,
        "accuracy": accuracy_subset,
        "balanced_accuracy": balanced_accuracy_subset,
        "precision": precision_subset,
        "recall": recall_subset,
        "f1": f1_subset,
        "tn": int(cm_subset[0, 0]),
        "fp": int(cm_subset[0, 1]),
        "fn": int(cm_subset[1, 0]),
        "tp": int(cm_subset[1, 1])
    })

modality_metrics = pd.DataFrame(
    modality_results
)

print("=" * 70)
print("MODALITY-SPECIFIC CROSS-MODAL METRICS")
print("=" * 70)

display(modality_metrics)

MODALITY-SPECIFIC CROSS-MODAL METRICS


,modality_combination,n_pairs,class_0,class_1,roc_auc,accuracy,balanced_accuracy,precision,recall,f1,tn,fp,fn,tp
0,Behavioral-Environmental,352,176,176,0.659607,0.610795,0.610795,0.606557,0.630682,0.618384,104,72,65,111
1,Genetic-Behavioral,48,24,24,0.807292,0.729167,0.729167,0.720000,0.750000,0.734694,17,7,6,18
2,Genetic-Environmental,48,24,24,0.501736,0.500000,0.500000,0.500000,0.541667,0.520000,11,13,11,13


In [36]:
# ============================================================
# CELL 33 — FINAL METRIC CONSISTENCY CHECK
# ============================================================

print("=" * 70)
print("FINAL METRIC CONSISTENCY CHECK")
print("=" * 70)

# Overall metrics
expected_overall = {
    "roc_auc": 0.6626674107,
    "accuracy": 0.6116071429,
    "balanced_accuracy": 0.6116071429,
    "precision": 0.6068376068,
    "recall": 0.6339285714,
    "f1": 0.6200873362
}

actual_overall = {
    "roc_auc": roc_auc,
    "accuracy": accuracy,
    "balanced_accuracy": balanced_accuracy,
    "precision": precision,
    "recall": recall,
    "f1": f1
}

for metric_name, expected_value in expected_overall.items():
    actual_value = actual_overall[metric_name]

    if not np.isclose(
        actual_value,
        expected_value,
        atol=1e-9
    ):
        raise ValueError(
            f"{metric_name} mismatch: "
            f"expected {expected_value}, "
            f"found {actual_value}"
        )

print("\nOverall metrics: PASSED")

# Dataset size
if len(prediction_table) != 448:
    raise ValueError(
        f"Expected 448 predictions, found {len(prediction_table)}"
    )

# Modality pair counts
expected_counts = {
    "Behavioral-Environmental": 352,
    "Genetic-Behavioral": 48,
    "Genetic-Environmental": 48
}

actual_counts = (
    prediction_table["modality_combination"]
    .value_counts()
    .to_dict()
)

if actual_counts != expected_counts:
    raise ValueError(
        f"Modality counts mismatch.\n"
        f"Expected: {expected_counts}\n"
        f"Found: {actual_counts}"
    )

print("Modality combination counts: PASSED")

print("\nFINAL METRIC CONSISTENCY CHECK: PASSED")

FINAL METRIC CONSISTENCY CHECK

Overall metrics: PASSED
Modality combination counts: PASSED

FINAL METRIC CONSISTENCY CHECK: PASSED


In [37]:
# ============================================================
# CELL 34 — SAVE CROSS-MODAL PREDICTIONS
# ============================================================

prediction_output_path = os.path.join(
    REPORT_ROOT,
    "05_04_cross_modal_predictions.csv"
)

prediction_table.to_csv(
    prediction_output_path,
    index=False
)

print("=" * 70)
print("CROSS-MODAL PREDICTIONS SAVED")
print("=" * 70)

print("\nFile:")
print(prediction_output_path)

print("\nRows:", len(prediction_table))
print("Columns:", len(prediction_table.columns))

print("\nSave status: PASSED")

CROSS-MODAL PREDICTIONS SAVED

File:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\05_multimodal_integration\reports\05_04_cross_modal_inference\05_04_cross_modal_predictions.csv

Rows: 448
Columns: 13

Save status: PASSED


In [38]:
# ============================================================
# CELL 35 — SAVE MODALITY COMBINATION METRICS
# ============================================================

modality_metrics_output_path = os.path.join(
    REPORT_ROOT,
    "05_04_modality_combination_metrics.csv"
)

modality_metrics.to_csv(
    modality_metrics_output_path,
    index=False
)

print("=" * 70)
print("MODALITY COMBINATION METRICS SAVED")
print("=" * 70)

print("\nFile:")
print(modality_metrics_output_path)

print("\nRows:", len(modality_metrics))
print("Columns:", len(modality_metrics.columns))

print("\nSave status: PASSED")

MODALITY COMBINATION METRICS SAVED

File:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\05_multimodal_integration\reports\05_04_cross_modal_inference\05_04_modality_combination_metrics.csv

Rows: 3
Columns: 14

Save status: PASSED


In [39]:
# ============================================================
# CELL 36 — CREATE FINAL 05.04 REPORT
# ============================================================

report = {
    "phase": "05.04",
    "title": "Cross-Modal Inference",
    "status": "PASSED",

    "objective": (
        "Run frozen Phase 03 cross-modal baseline inference "
        "on the Phase 02.08 representation-aware validation pairs."
    ),

    "training_performed": False,
    "checkpoint_modified": False,

    "validation_dataset": {
        "source_embeddings_shape": list(source_embeddings.shape),
        "target_embeddings_shape": list(target_embeddings.shape),
        "pair_labels_shape": list(pair_labels.shape),
        "total_pairs": int(len(pair_labels)),
        "class_0": int(np.sum(pair_labels == 0)),
        "class_1": int(np.sum(pair_labels == 1))
    },

    "pair_feature_construction": {
        "source_dimension": 32,
        "target_dimension": 32,
        "absolute_difference_dimension": 32,
        "elementwise_product_dimension": 32,
        "total_dimension": 128,
        "formula": "[S, T, |S-T|, S*T]"
    },

    "baseline_model": {
        "input_dimension": 128,
        "hidden_dimensions": [128, 64],
        "output_dimension": 1,
        "parameter_count": 24833,
        "frozen": True
    },

    "overall_metrics": {
        "roc_auc": float(roc_auc),
        "accuracy": float(accuracy),
        "balanced_accuracy": float(balanced_accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "confusion_matrix": cm.tolist()
    },

    "modality_combination_metrics": (
        modality_metrics.to_dict(orient="records")
    ),

    "interpretation": {
        "output_type": (
            "cross-modal relationship probability"
        ),
        "clinical_diagnostic_probability": False,
        "note": (
            "The model output represents the probability assigned "
            "to the cross-modal pair relationship class. It should "
            "not be interpreted as an individual ASD diagnostic "
            "probability."
        )
    },

    "artifacts": {
        "prediction_csv": prediction_output_path,
        "modality_metrics_csv": modality_metrics_output_path
    }
}

report_output_path = os.path.join(
    REPORT_ROOT,
    "05_04_cross_modal_inference_report.json"
)

with open(
    report_output_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        report,
        f,
        indent=4
    )

print("=" * 70)
print("05.04 REPORT SAVED")
print("=" * 70)

print("\nFile:")
print(report_output_path)

print("\nReport status: PASSED")

05.04 REPORT SAVED

File:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\05_multimodal_integration\reports\05_04_cross_modal_inference\05_04_cross_modal_inference_report.json

Report status: PASSED


In [40]:
# ============================================================
# CELL 37 — FINAL ARTIFACT VERIFICATION
# ============================================================

print("=" * 70)
print("FINAL 05.04 ARTIFACT VERIFICATION")
print("=" * 70)

required_outputs = {
    "Prediction CSV": prediction_output_path,
    "Modality metrics CSV": modality_metrics_output_path,
    "Inference report": report_output_path
}

for name, path in required_outputs.items():

    if not os.path.exists(path):
        raise FileNotFoundError(
            f"{name} was not created:\n{path}"
        )

    print(f"\n[PASSED] {name}")
    print(path)

# Reload CSV
saved_predictions = pd.read_csv(
    prediction_output_path
)

saved_modality_metrics = pd.read_csv(
    modality_metrics_output_path
)

# Reload JSON
with open(
    report_output_path,
    "r",
    encoding="utf-8"
) as f:
    saved_report = json.load(f)

# Verify row counts
if len(saved_predictions) != 448:
    raise ValueError(
        "Saved prediction CSV does not contain 448 rows."
    )

if len(saved_modality_metrics) != 3:
    raise ValueError(
        "Saved modality metrics CSV does not contain 3 rows."
    )

if saved_report["status"] != "PASSED":
    raise ValueError(
        "Saved report status is not PASSED."
    )

print("\nPrediction CSV rows:", len(saved_predictions))
print("Modality metrics rows:", len(saved_modality_metrics))
print("Report status:", saved_report["status"])

print("\nFINAL 05.04 ARTIFACT VERIFICATION: PASSED")

FINAL 05.04 ARTIFACT VERIFICATION

[PASSED] Prediction CSV
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\05_multimodal_integration\reports\05_04_cross_modal_inference\05_04_cross_modal_predictions.csv

[PASSED] Modality metrics CSV
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\05_multimodal_integration\reports\05_04_cross_modal_inference\05_04_modality_combination_metrics.csv

[PASSED] Inference report
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\05_multimodal_integration\reports\05_04_cross_modal_inference\05_04_cross_modal_inference_report.json

Prediction CSV rows: 448
Modality metrics rows: 3
Report status: PASSED

FINAL 05.04 ARTIFACT VERIFICATION: PASSED


In [42]:
# # 05.04 Cross-Modal Inference — Final Status

# ## Status: PASSED

# ### Inference Pipeline

# Phase 02.08 representation-aware validation pairs  
# → 32-D source representation  
# → 32-D target representation  
# → `[S, T, |S-T|, S×T]`  
# → 128-D pair representation  
# → Frozen Phase 03 baseline  
# → Logit  
# → Sigmoid  
# → Cross-modal relationship probability

# ### Validation Dataset

# - Total pairs: 448
# - Class 0: 224
# - Class 1: 224

# ### Overall Results

# - ROC-AUC: 0.6626674107
# - Accuracy: 0.6116071429
# - Balanced Accuracy: 0.6116071429
# - Precision: 0.6068376068
# - Recall: 0.6339285714
# - F1: 0.6200873362

# Confusion Matrix:

# [[132, 92],
#  [82, 142]]

# ### Modality Combinations

# - Behavioral–Environmental: 352 pairs
# - Genetic–Behavioral: 48 pairs
# - Genetic–Environmental: 48 pairs

# ### Integrity

# - No training performed
# - No checkpoint modified
# - Frozen Phase 03 baseline used
# - 448/448 validation pairs processed
# - Phase 04 baseline metrics reproduced exactly

# ### Important Interpretation

# The model output is a cross-modal relationship probability.

# It is not an individual clinical ASD diagnostic probability.

# ## Official Outputs

# `reports/05_04_cross_modal_inference/`

# - `05_04_cross_modal_predictions.csv`
# - `05_04_modality_combination_metrics.csv`
# - `05_04_cross_modal_inference_report.json`